# Imports

In [ ]:
import json
import pandas as pd
import numpy as np

# Constants

In [ ]:
ORIGINAL_TEST_FILE =  "df_test.json"
HIGH_CONFIDENCE_FILE = "high_confidence_annotation_examples.json"
PREV_ANNOT_TEST_FILE = "full_labeled_final_dataset.jsonl"
Y_TRUE_FILE = "y_true.json"
Y_PRED_FILES = ["y_pred_bertin.json", "y_pred_xlm_roberta.json"]

In [ ]:
N_EXAMPLES = 500

In [ ]:
OUTPUT_FILE = ""

# Utils

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

# Execution

## Load data

In [ ]:
df_test = pd.read_json(ORIGINAL_TEST_FILE, lines=True)
df_test.head(5)

In [ ]:
df_prev_annot = pd.read_json(PREV_ANNOT_TEST_FILE, lines=True)
df_prev_annot["was_annotated"] = True
df_prev_annot.head(5)

In [ ]:
df_high_confidence = pd.read_json(HIGH_CONFIDENCE_FILE, lines=True)
df_high_confidence["was_annotated"] = True
df_high_confidence.head(5)

In [ ]:
df_prev_annot = pd.concat([df_prev_annot, df_high_confidence])

In [ ]:
y_true = np.array(load_json(Y_TRUE_FILE))

In [ ]:
y_preds = [
    load_json(file)
    for file in Y_PRED_FILES
]
# Apply softmax for probas
y_preds = np.array([
    np.exp(y_pred)/np.reshape(np.sum(np.exp(y_pred), axis=1), (len(y_pred), 1))
    for y_pred in y_preds
])

## Visualizing confidence values for all predictions
Averaging maximum confidence between models, without considering if the label matches.

In [ ]:
df_confidences = pd.Series(y_preds.max(axis=2).mean(axis=0))
df_confidences.plot(kind="hist", bins=100)

In [ ]:
df_confidences.describe(percentiles=np.arange(0,1, 0.1))

## Find indexes where predictions match the original label

In [ ]:
y_true_label = np.argmax(y_true, axis=1)
y_pred_labels = np.argmax(y_preds, axis=2)
valid_example_mask = np.ones(y_true_label.shape, dtype=bool)
for model_pred_index in range(y_pred_labels.shape[0]):
    are_valid_examples = y_true_label == y_pred_labels[model_pred_index]
    valid_example_mask = valid_example_mask & are_valid_examples

In [ ]:
valid_example_mask.sum()

In [ ]:
y_pred_confidence = np.mean(np.max(y_preds, axis=2), axis=0)
y_pred_confidence.shape

In [ ]:
df_test["confidence"] = y_pred_confidence
df_test = df_test.loc[valid_example_mask].copy(deep=True)

In [ ]:
len(df_test)

## Visualizing confidence values for matched labels

In [ ]:
df_test["confidence"].plot(kind="hist", bins=100)

In [ ]:
df_test["confidence"].describe(percentiles=np.arange(0,1, 0.1))

## Filtering samples with medium confidence only

In [ ]:
medium_confidence_bottom = 0.6
medium_confidence_top = 0.75

In [ ]:
is_medium_confidence = (
    (df_test["confidence"] >= medium_confidence_bottom)
    & (df_test["confidence"] <= medium_confidence_top)
)
df_test = df_test[is_medium_confidence]

In [ ]:
len(df_test)

## Remove examples that were already used for annotation

In [ ]:
df_test = df_test.merge(
    df_prev_annot[["sentence_1", "sentence_2", "was_annotated"]],
    on=[
        "sentence_1",
        "sentence_2"
    ],
    how="left"
)

In [ ]:
len(df_test)

In [ ]:
df_test = df_test[
    df_test["was_annotated"].isnull()
].copy(deep=True)

In [ ]:
df_test.drop(columns="was_annotated", inplace=True)

In [ ]:
len(df_test)

## Sort by confidence

In [ ]:
df_test.sort_values("confidence", ascending=False, inplace=True)

## Filter top examples by confidence, proportional to "dataset,connector_type" example count, but keep each dataset balanced

In [ ]:
top_examples_balanced = []
for dataset, df_dataset in df_test.groupby("dataset"):
    max_class_len_dataset = df_dataset["connector_type"].value_counts().min()
    for label, df_label in df_dataset.groupby("connector_type"):
        n_examples_selected =  int(
            N_EXAMPLES
            * min(len(df_label), max_class_len_dataset)
            / len(df_test)
        )

        top_examples_balanced.append(df_label.sample(n_examples_selected).copy(deep=True))
df_top_examples_balanced = pd.concat(top_examples_balanced)

In [ ]:
len(df_top_examples_balanced)

In [ ]:
df_top_examples_balanced.confidence.describe()

In [ ]:
df_top_examples_balanced["connector_type"].value_counts()

In [ ]:
df_top_examples_balanced.groupby(["dataset", "connector_type"])["connector_type"].count().to_dict()

## Save results to jsonl

In [ ]:
df_top_examples_balanced.to_json("medium_confidence_annotation_examples.json", lines=True, orient="records")